# Part 5 — Judging the Judge: Measuring and Correcting LLM-Judge Bias

*Evals from First Principles, Part 5.*

An LLM judge is itself a model, so it has systematic, MEASURABLE biases. This notebook builds a deterministic mock judge with two baked-in flaws: a POSITION bias (it gives the answer shown first, in slot A, a fixed bonus) and a VERBOSITY bias (it rewards longer answers regardless of quality). We measure each on a dozen paired A-vs-B comparisons, then CORRECT the position bias by running both orderings and averaging, and watch the disagreement-with-truth rate fall.

The judge scores each answer as:

    score = quality + (POS_BIAS if shown in slot A else 0) + VERB_PER_WORD * words

A fair judge would use `quality` alone. Ours does not.

Pure Python, no imports beyond the standard library's `namedtuple`, fully offline and deterministic — every number below is reproducible on a laptop with no network and no API key.

(Self-preference — a bonus the judge hands to answers from its own model — is the exact same recipe with a different trigger; measure and correct it the same way. We keep the runnable demo to position + verbosity to stay concrete.)

Companion script: `judge_bias.py`

## Step 1 — The judge's two baked-in biases, and the paired comparisons

`POS_BIAS` is a full quality-point handed to whichever answer sits in slot A, no matter its content. `VERB_PER_WORD` hands extra credit per word, so a wordier answer creeps ahead regardless of quality.

Each pair pits system X against system Y on one prompt. An answer is just its `(quality, words)`: `quality` is the honest score a fair judge would use, `words` is its length. The TRUE winner is simply the higher-quality answer — that is the ground truth we will compare the judge against.

In [ ]:
from collections import namedtuple

POS_BIAS = 1.0
VERB_PER_WORD = 0.02

Pair = namedtuple("Pair", "id xq xw yq yw")

PAIRS = [
    Pair("p01", 5, 40, 6, 20),   # Y better, X wordier  -> position tips it to X
    Pair("p02", 6, 50, 7, 25),   # Y better, X wordier
    Pair("p03", 3, 45, 4, 30),   # Y better, X wordier
    Pair("p04", 7, 35, 8, 25),   # Y better, X wordier
    Pair("p05", 9, 30, 4, 40),   # X clearly better
    Pair("p06", 8, 20, 5, 30),   # X clearly better
    Pair("p07", 3, 30, 8, 20),   # Y clearly better
    Pair("p08", 4, 25, 7, 35),   # Y clearly better
    Pair("p09", 7, 25, 6, 40),   # X better, Y wordier  -> order-sensitive
    Pair("p10", 8, 20, 7, 45),   # X better, Y wordier  -> order-sensitive
    Pair("p11", 5, 70, 6, 15),   # Y better, X much wordier -> verbosity wins
    Pair("p12", 6, 35, 5, 25),   # X clearly better
]

print(f"Judge biases baked in:  slot-A bonus = +{POS_BIAS:.2f} quality-points,"
      f"  +{VERB_PER_WORD:.2f}/word")
print(f"Paired comparisons:     {len(PAIRS)} prompts, system X vs system Y")
# Judge biases baked in:  slot-A bonus = +1.00 quality-points,  +0.02/word
# Paired comparisons:     12 prompts, system X vs system Y

## Step 2 — The judge's scoring function

`answer_score` is the mock judge's biased score for one answer sitting in one slot: `quality`, plus `POS_BIAS` if it happens to be in slot A, plus `VERB_PER_WORD` for every word. `judge` compares the two slots and returns the winning SLOT, `'A'` or `'B'` — note ties go to A, which is the position bias showing up a second time.

In [ ]:
def answer_score(quality, words, in_slot_a):
    """The mock judge's biased score for one answer in one slot."""
    bonus = POS_BIAS if in_slot_a else 0.0
    return quality + bonus + VERB_PER_WORD * words


def judge(slot_a, slot_b):
    """Given (quality, words) for the answer in slot A and in slot B,
    return 'A' or 'B' - the winning SLOT. Ties go to A (position bias again)."""
    sa = answer_score(slot_a[0], slot_a[1], in_slot_a=True)
    sb = answer_score(slot_b[0], slot_b[1], in_slot_a=False)
    return "A" if sa >= sb else "B"

## Step 3 — From slots to systems, and the ground truth

`judge` only knows about slots A and B; `verdict` translates that into a winning SYSTEM ('X' or 'Y') for a given ordering. `true_winner` is the ground truth: whichever system actually has the higher `quality`, independent of slot or length. `corrected_winner` is the fix we will use later — run both orderings and average each system's two scores, so the slot-A bonus cancels (every answer sits in slot A exactly once). `rate` is just a guarded division for reporting fractions.

In [ ]:
def verdict(pair, x_first):
    """Winning SYSTEM ('X' or 'Y') when X is (or is not) shown first in slot A."""
    x = (pair.xq, pair.xw)
    y = (pair.yq, pair.yw)
    if x_first:
        return "X" if judge(x, y) == "A" else "Y"
    else:
        return "Y" if judge(y, x) == "A" else "X"


def true_winner(pair):
    """What a fair judge would say: the higher-quality answer wins."""
    return "X" if pair.xq > pair.yq else "Y"


def corrected_winner(pair):
    """Correction: run BOTH orderings and average each system's two scores.
    Every answer sits in slot A exactly once, so the position bonus cancels."""
    x_avg = (answer_score(pair.xq, pair.xw, True)
             + answer_score(pair.xq, pair.xw, False)) / 2
    y_avg = (answer_score(pair.yq, pair.yw, True)
             + answer_score(pair.yq, pair.yw, False)) / 2
    return "X" if x_avg >= y_avg else "Y"


def rate(count, total):
    return count / total if total else 0.0

## Step 4 — Measuring position bias: swap the order, count the flips

For each of the 12 pairs we ask the judge twice: once with X shown first, once with Y shown first. If the naive (X-always-first) judge is fair, swapping the order should never change which system wins. Every row where it DOES change is a position-bias flip. We also record the true winner alongside, so we can see which flips actually cost us a correct verdict.

In [ ]:
header = f"{'pair':>4}  {'X(q,words)':>11}  {'Y(q,words)':>11}  {'X-1st':>5}  {'Y-1st':>5}  {'flip':>4}  {'true':>4}"
print(header)
flips = 0
naive_wrong = 0
for p in PAIRS:
    x_first = verdict(p, x_first=True)     # naive: X always shown first
    y_first = verdict(p, x_first=False)    # the swapped ordering
    flipped = x_first != y_first
    flips += flipped
    naive_wrong += (x_first != true_winner(p))
    print(f"{p.id:>4}  {f'({p.xq},{p.xw})':>11}  {f'({p.yq},{p.yw})':>11}"
          f"  {x_first:>5}  {y_first:>5}  {('YES' if flipped else '.'):>4}"
          f"  {true_winner(p):>4}")
pos_rate = rate(flips, len(PAIRS))
print(f"\n  position-bias rate = flips / pairs = {flips}/{len(PAIRS)} = {pos_rate:.2f}")
print(f"  a fair judge would flip 0 times; ours flips on every close call.")
# pair   X(q,words)   Y(q,words)  X-1st  Y-1st  flip  true
#  p01       (5,40)       (6,20)      X      Y   YES     Y
#  p02       (6,50)       (7,25)      X      Y   YES     Y
#  p03       (3,45)       (4,30)      X      Y   YES     Y
#  p04       (7,35)       (8,25)      X      Y   YES     Y
#  p05       (9,30)       (4,40)      X      X     .     X
#  p06       (8,20)       (5,30)      X      X     .     X
#  p07       (3,30)       (8,20)      Y      Y     .     Y
#  p08       (4,25)       (7,35)      Y      Y     .     Y
#  p09       (7,25)       (6,40)      X      Y   YES     X
#  p10       (8,20)       (7,45)      X      Y   YES     X
#  p11       (5,70)       (6,15)      X      Y   YES     Y
#  p12       (6,35)       (5,25)      X      X     .     X
#
#   position-bias rate = flips / pairs = 7/12 = 0.58
#   a fair judge would flip 0 times; ours flips on every close call.

## Step 5 — Measuring verbosity bias: equal quality, does longer still win?

To isolate length from position, we build 5 new pairs whose two answers are TIED in `quality` and differ ONLY in word count, and we score them with `corrected_winner` — which already cancels the position bonus. The only thing left that can sway a tie is verbosity. If the judge were fair, it would call every one of these pairs a coin flip; instead, watch the longer answer win every single time.

In [ ]:
print("   Order-averaged verdicts on 5 pairs whose two answers are TIED in")
print("   quality and differ only in length:")
probes = [
    Pair("q01", 5, 50, 5, 20),   # X longer
    Pair("q02", 6, 20, 6, 45),   # Y longer
    Pair("q03", 4, 55, 4, 25),   # X longer
    Pair("q04", 7, 30, 7, 60),   # Y longer
    Pair("q05", 3, 40, 3, 15),   # X longer
]
print(f"   {'pair':>4}  {'X words':>7}  {'Y words':>7}  {'longer':>6}  {'winner':>6}")
longer_wins = 0
for p in probes:
    w = corrected_winner(p)
    longer = "X" if p.xw > p.yw else "Y"
    longer_wins += (w == longer)
    print(f"   {p.id:>4}  {p.xw:>7}  {p.yw:>7}  {longer:>6}  {w:>6}")
verb_rate = rate(longer_wins, len(probes))
print(f"\n  verbosity-bias rate = longer-wins / pairs = {longer_wins}/{len(probes)} = {verb_rate:.2f}")
print(f"  quality is tied, yet the longer answer wins every time.")
#    pair  X words  Y words  longer  winner
#     q01       50       20       X       X
#     q02       20       45       Y       Y
#     q03       55       25       X       X
#     q04       30       60       Y       Y
#     q05       40       15       X       X
#
#   verbosity-bias rate = longer-wins / pairs = 5/5 = 1.00
#   quality is tied, yet the longer answer wins every time.

## Step 6 — Correcting position bias, and what survives

Now we run `corrected_winner` on the original 12 pairs and compare it to `true_winner`. If order-averaging truly cancels the slot-A bonus, the corrected disagreement rate should drop far below the naive (X-always-first) rate — and the position-bias rate on the corrected judge should hit exactly zero, since order can no longer flip a verdict at all. Whatever disagreement remains has to come from somewhere else: verbosity.

In [ ]:
corr_wrong = 0
corr_flips = 0  # a "flip" needs the verdict to depend on order; it no longer does
for p in PAIRS:
    c = corrected_winner(p)
    corr_wrong += (c != true_winner(p))
naive_rate = rate(naive_wrong, len(PAIRS))
corr_rate = rate(corr_wrong, len(PAIRS))
print(f"  naive judge (X always first):  disagreement with truth"
      f" = {naive_wrong}/{len(PAIRS)} = {naive_rate:.2f}")
print(f"  corrected (both orders avg):   disagreement with truth"
      f" = {corr_wrong}/{len(PAIRS)} = {corr_rate:.2f}")
print(f"  corrected position-bias rate:  {corr_flips}/{len(PAIRS)} = "
      f"{rate(corr_flips, len(PAIRS)):.2f}  (order can no longer flip a verdict)")
print(f"\n  -> Averaging killed the position bias, cutting disagreement from")
print(f"     {naive_wrong}/{len(PAIRS)} to {corr_wrong}/{len(PAIRS)}. The lone survivor is a"
      f" VERBOSITY error:")
print(f"     order-averaging fixes WHERE an answer sits, not HOW LONG it is.")
#   naive judge (X always first):  disagreement with truth = 5/12 = 0.42
#   corrected (both orders avg):   disagreement with truth = 1/12 = 0.08
#   corrected position-bias rate:  0/12 = 0.00  (order can no longer flip a verdict)
#
#   -> Averaging killed the position bias, cutting disagreement from
#      5/12 to 1/12. The lone survivor is a VERBOSITY error:
#      order-averaging fixes WHERE an answer sits, not HOW LONG it is.

## Recap

- An LLM judge is a model, so it has the same kind of systematic biases any model has — and those biases are MEASURABLE the same way any model's errors are: run controlled probes and count.
- **Position bias**: swapping which answer is shown first flipped the verdict on 7 of 12 pairs (0.58). A fair judge would never flip on order alone.
- **Verbosity bias**: on 5 pairs with tied quality and different lengths, the longer answer won every time (1.00) — length was standing in for quality.
- **Correction**: running both orderings and averaging cancels the position bonus exactly, because every answer sits in slot A once. Disagreement with the true winner fell from 5/12 (0.42) to 1/12 (0.08), and the corrected judge's position-bias rate is exactly 0/12.
- The one survivor after correction is a verbosity error: order-averaging fixes WHERE an answer sits, not HOW LONG it is. Different biases need different fixes.

Next: what happens when you need to measure not one judgment, but a whole trajectory of tool calls and decisions an agent makes on the way to an answer.